# Causal Torque-Direction Steering (A100)

Tests whether changing only the frozen net-torque readout direction at the selected decoder layer causally changes the model's answer.

[实验设置与运行说明](../../../docs/experiment_notes/03_section44_readout_correction/03dd8a719cc48736.md)


In [1]:
# Run this helper once. Every later code cell uses %%cell_progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 16 tracked cells.')


Initialize progress display:   0%|          | 0/1 [00:00<?, ?cell/s]

Cell progress helper ready: 16 tracked cells.


## 1. Setup

The protocol lock is written before probe_test examples are selected or the model is loaded.


In [2]:
%%cell_progress 1/16 Mount Drive and locate VisualTorque
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


Mount Drive and locate VisualTorque:   0%|          | 0/16 [00:00<?, ?cell/s]

Mounted at /content/drive
Project: /content/drive/MyDrive/VisualTorque


In [3]:
%%cell_progress 2/16 Install and verify the multimodal environment
import importlib
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)
cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34', 'tqdm>=4.66',
    'scikit-learn>=1.5', 'pandas>=2.2', 'matplotlib>=3.8',
])
cell_stage('Installing VisualTorque toolkit')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', str(TOOLKIT_ROOT)
])
SRC_ROOT = TOOLKIT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
importlib.invalidate_caches()

import torch
import torchvision
import visual_torque_toolkit
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import (
    HuggingFaceMultimodalAdapter,
)
required_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(
    hasattr(HuggingFaceMultimodalAdapter, name)
    for name in required_features
)
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('Toolkit:', visual_torque_toolkit.__file__)


Install and verify the multimodal environment:   6%|6         | 1/16 [00:00<?, ?cell/s]

torch: 2.11.0+cu128 | CUDA: 12.8
torchvision: 0.26.0+cu128
Toolkit: /content/drive/MyDrive/VisualTorque/toolkit/src/visual_torque_toolkit/__init__.py


In [4]:
%%cell_progress 3/16 Verify A100 and Hugging Face access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Select an A100 GPU runtime.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(
    torch.cuda.get_device_properties(0).total_memory / 2**30, 1
))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Public checkpoints may still work.')
except Exception as exc:
    print('Hugging Face login skipped:', exc)


Verify A100 and Hugging Face access:  12%|#2        | 2/16 [00:00<?, ?cell/s]

NVIDIA A100-SXM4-40GB
CUDA memory (GB): 39.5
Hugging Face login skipped: Secret HF_TOKEN does not exist.


In [5]:
%%cell_progress 4/16 Lock model and causal intervention choices
cell_stage('Writing the causal follow-up protocol before test selection')
import hashlib
import json
from datetime import datetime, timezone

# Run 1 (default).
MODEL_ID = 'Qwen/Qwen3.5-9B'
BATCH_SIZE = 4

# Run 2: uncomment both lines below in a fresh runtime.
# MODEL_ID = 'google/gemma-4-12B-it'
# BATCH_SIZE = 2

MODEL_SPECS = {
    'Qwen/Qwen3.5-9B': {
        'slug': 'Qwen__Qwen3.5-9B',
        'expected_layer': 13,
        'expected_alpha': 0.01,
    },
    'google/gemma-4-12B-it': {
        'slug': 'google__gemma-4-12B-it',
        'expected_layer': 7,
        'expected_alpha': 100.0,
    },
}
MODEL_DTYPE = 'bfloat16'
DEVICE_MAP = 'auto'
SEED = 20260820
DOSE_MULTIPLIERS = (-2.0, -1.0, 0.0, 1.0, 2.0)
DOSE_UNIT_POLICY = 'half_training_net_torque_standard_deviation'
ORTHOGONAL_DIRECTIONS = 24
ORTHOGONAL_CONTROL_SCENES = 72
BOOTSTRAP_REPLICATES = 5000
SCENE_CAPS = {
    'distance': 54,
    'conflict_distance': 42,
    'conflict_balance': 12,
    'balance': 6,
    'weight': 40,
    'conflict_weight': 40,
    'compatible_weight_distance': 40,
    'multi_stack': 70,
}

spec = MODEL_SPECS[MODEL_ID]
MODEL_ROOT = PROJECT_ROOT / 'results' / spec['slug']
DISCOVERY_ROOT = MODEL_ROOT / 'activation_discovery_v1'
CONFIRMATORY_ROOT = MODEL_ROOT / 'activation_confirmatory_v1'
SPECIFICITY_ROOT = PROJECT_ROOT / 'results' / 'representation_specificity_v1'
OUTPUT_ROOT = MODEL_ROOT / 'causal_torque_intervention_v1'
SHARD_DIR = OUTPUT_ROOT / 'shards'
ANALYSIS_DIR = OUTPUT_ROOT / 'analysis'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SHARD_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
PROTOCOL_LOCK_PATH = OUTPUT_ROOT / 'protocol_lock.json'

protocol_identity = {
    'schema_version': 'causal-torque-steering-protocol-1.0',
    'analysis_status': 'frozen_exploratory_followup_after_probe_test_opened',
    'model_id': MODEL_ID,
    'batch_size': BATCH_SIZE,
    'expected_layer': spec['expected_layer'],
    'expected_alpha': spec['expected_alpha'],
    'representation_position': 'selected_layer_final_prompt_token',
    'intervention_geometry': (
        'minimum StandardScaler-coordinate L2 perturbation for a '
        'prespecified frozen-probe torque shift'
    ),
    'dose_multipliers': list(DOSE_MULTIPLIERS),
    'dose_unit_policy': DOSE_UNIT_POLICY,
    'scene_caps': SCENE_CAPS,
    'one_deterministic_option_permutation_per_scene': True,
    'orthogonal_directions': ORTHOGONAL_DIRECTIONS,
    'orthogonal_control_scenes': ORTHOGONAL_CONTROL_SCENES,
    'orthogonal_control_geometry': (
        'probe-orthogonal in standardized coordinates with matched '
        'standardized endpoint norm'
    ),
    'neighbor_layer_controls': [-1, 1],
    'primary_answer_metric': 'semantic_logprob_left_minus_right',
    'primary_causal_tests': [
        'positive mean endpoint margin effect with scene-bootstrap CI',
        'positive toward-gold versus away-from-gold margin effect',
        'torque endpoint effect exceeds 24 orthogonal direction effects',
    ],
    'causal_support_rule': (
        'endpoint-effect CI lower bound > 0; toward-vs-away CI lower '
        'bound > 0; orthogonal empirical p <= 0.05'
    ),
    'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    'interpretation_limit': (
        'Directional steering can establish causal influence of the '
        'readout direction, not natural mediation, uniqueness, or an '
        'on-manifold torque circuit.'
    ),
    'seed': SEED,
}
if PROTOCOL_LOCK_PATH.is_file():
    previous = json.loads(PROTOCOL_LOCK_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == protocol_identity, (
        'Existing causal protocol differs. Use a new versioned output directory.'
    )
else:
    temporary = PROTOCOL_LOCK_PATH.with_suffix('.json.tmp')
    temporary.write_text(
        json.dumps({
            'identity': protocol_identity,
            'locked_at_utc': datetime.now(timezone.utc).isoformat(),
        }, indent=2, ensure_ascii=False) + '\n',
        encoding='utf-8',
    )
    temporary.replace(PROTOCOL_LOCK_PATH)
print('Model:', MODEL_ID)
print('Protocol:', PROTOCOL_LOCK_PATH)
print('Planned scenes:', sum(SCENE_CAPS.values()))


Lock model and causal intervention choices:  19%|#8        | 3/16 [00:00<?, ?cell/s]

Model: Qwen/Qwen3.5-9B
Protocol: /content/drive/MyDrive/VisualTorque/results/Qwen__Qwen3.5-9B/causal_torque_intervention_v1/protocol_lock.json
Planned scenes: 304


In [6]:
%%cell_progress 5/16 Prepare Dataset v1 on local SSD
cell_stage('Checking local and Drive dataset caches')
from zipfile import ZipFile

REQUIRED_DATASET_FILES = {
    'dataset_config.json', 'scenes.jsonl', 'counterfactual_pairs.jsonl',
    'questions.jsonl', 'splits.json', 'statistics.json',
}
DATASET_DIR = Path('/content/visualtorque_dataset_v1')
DATASET_ARCHIVE = PROJECT_ROOT / 'data' / 'dataset_v1.zip'
LEGACY_DATASET_DIR = PROJECT_ROOT / 'data' / 'dataset_v1'

def dataset_complete(path):
    return path.is_dir() and all(
        (path / name).is_file() for name in REQUIRED_DATASET_FILES
    )

if not dataset_complete(DATASET_DIR):
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if DATASET_ARCHIVE.is_file():
        cell_stage('Extracting Dataset v1 archive')
        with ZipFile(DATASET_ARCHIVE, 'r') as bundle:
            members = [
                member for member in bundle.infolist()
                if not member.is_dir()
            ]
            root = DATASET_DIR.resolve()
            for member in tqdm(
                members, desc='Extracting Dataset v1', unit='file'
            ):
                target = (DATASET_DIR / member.filename).resolve()
                if root not in target.parents:
                    raise ValueError(f'Unsafe archive path: {member.filename}')
                bundle.extract(member, DATASET_DIR)
    elif dataset_complete(LEGACY_DATASET_DIR):
        DATASET_DIR = LEGACY_DATASET_DIR
    else:
        raise FileNotFoundError(
            'Run the Dataset v1 build/cache cell in the behavior notebook first.'
        )
assert dataset_complete(DATASET_DIR)
print('Dataset:', DATASET_DIR)


Prepare Dataset v1 on local SSD:  25%|##5       | 4/16 [00:00<?, ?cell/s]

Extracting Dataset v1:   0%|          | 0/2106 [00:00<?, ?file/s]

Dataset: /content/visualtorque_dataset_v1


In [7]:
%%cell_progress 6/16 Select frozen causal scenes and one option permutation
cell_stage('Loading probe_test image-rule outcome examples')
from collections import Counter, defaultdict

from visual_torque_toolkit.behavior import (
    BehaviorRunConfig, PromptCondition, load_behavior_examples,
)

def read_jsonl(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

discovery_config_path = DISCOVERY_ROOT / 'capture_config.json'
discovery_selection_path = (
    DISCOVERY_ROOT / 'probe_results' / 'discovery_selection.json'
)
discovery_activation_path = DISCOVERY_ROOT / 'scene_mean_activations.pt'
confirmatory_summary_path = (
    CONFIRMATORY_ROOT / 'confirmatory_results' / 'confirmatory_summary.json'
)
specificity_summary_path = SPECIFICITY_ROOT / 'specificity_summary.json'
for required in (
    discovery_config_path, discovery_selection_path,
    discovery_activation_path, confirmatory_summary_path,
    specificity_summary_path,
):
    assert required.is_file(), f'Missing prerequisite: {required}'

discovery_config = json.loads(
    discovery_config_path.read_text(encoding='utf-8')
)
discovery_identity = discovery_config['identity']
MODEL_REVISION = str(discovery_identity['resolved_revision'])
example_config = BehaviorRunConfig(
    dataset_dir=DATASET_DIR,
    output_dir=Path('/content/unused_causal_output'),
    model_id=MODEL_ID,
    model_revision=MODEL_REVISION,
    conditions=(PromptCondition.IMAGE_RULE,),
    question_kinds=('outcome',),
    splits=('probe_test',),
    batch_size=BATCH_SIZE,
    generation_every=0,
    seed=SEED,
)
all_examples = tuple(load_behavior_examples(example_config))
examples_by_scene = defaultdict(list)
for example in all_examples:
    examples_by_scene[str(example.scene_id)].append(example)
assert set(len(values) for values in examples_by_scene.values()) == {6}

scene_records = read_jsonl(DATASET_DIR / 'scenes.jsonl')
scene_by_id = {str(scene['scene_id']): scene for scene in scene_records}
test_scenes = [scene for scene in scene_records if scene['split'] == 'probe_test']

def stable_key(value):
    return hashlib.sha256(f'{SEED}:{value}'.encode('utf-8')).hexdigest()

def balanced_take(scenes, cap):
    groups = defaultdict(list)
    for scene in scenes:
        groups[str(scene['derived']['outcome'])].append(scene)
    for values in groups.values():
        values.sort(key=lambda scene: stable_key(scene['scene_id']))
    outcomes = sorted(groups)
    selected = []
    cursor = {outcome: 0 for outcome in outcomes}
    while len(selected) < min(cap, len(scenes)):
        changed = False
        for outcome in outcomes:
            index = cursor[outcome]
            if index < len(groups[outcome]) and len(selected) < cap:
                selected.append(groups[outcome][index])
                cursor[outcome] += 1
                changed = True
        if not changed:
            break
    return selected

selected_scenes = []
for problem_type, cap in SCENE_CAPS.items():
    candidates = [
        scene for scene in test_scenes
        if scene['problem_type'] == problem_type
    ]
    selected_scenes.extend(balanced_take(candidates, cap))
assert len(selected_scenes) == sum(SCENE_CAPS.values())
selected_scene_ids = {str(scene['scene_id']) for scene in selected_scenes}
assert len(selected_scene_ids) == len(selected_scenes)

selected_examples = []
for scene in sorted(selected_scenes, key=lambda value: value['scene_id']):
    scene_id = str(scene['scene_id'])
    variant_index = int(stable_key('variant:' + scene_id)[:8], 16) % 6
    variant = f'permutation_{variant_index}'
    matches = [
        example for example in examples_by_scene[scene_id]
        if example.variant == variant
    ]
    assert len(matches) == 1
    selected_examples.append(matches[0])

def round_robin_problem_types(examples, count):
    groups = defaultdict(list)
    for example in examples:
        groups[example.problem_type].append(example)
    for values in groups.values():
        values.sort(key=lambda item: stable_key('control:' + item.scene_id))
    problem_types = sorted(groups)
    selected = []
    cursor = {name: 0 for name in problem_types}
    while len(selected) < count:
        changed = False
        for name in problem_types:
            index = cursor[name]
            if index < len(groups[name]) and len(selected) < count:
                selected.append(groups[name][index])
                cursor[name] += 1
                changed = True
        if not changed:
            break
    return selected

control_examples = round_robin_problem_types(
    selected_examples, ORTHOGONAL_CONTROL_SCENES
)
control_scene_ids = {example.scene_id for example in control_examples}
assert len(control_scene_ids) == ORTHOGONAL_CONTROL_SCENES

selection_manifest = []
for example in selected_examples:
    scene = scene_by_id[str(example.scene_id)]
    selection_manifest.append({
        'schema_version': 'causal-selection-1.0',
        'example_id': example.example_id,
        'scene_id': str(example.scene_id),
        'variant': example.variant,
        'problem_type': example.problem_type,
        'gold_outcome': str(example.gold_semantic),
        'net_torque': int(scene['derived']['net_torque']),
        'orthogonal_control_subset': example.scene_id in control_scene_ids,
    })
manifest_path = OUTPUT_ROOT / 'selection_manifest.jsonl'
temporary_manifest = manifest_path.with_suffix('.jsonl.tmp')
with temporary_manifest.open('w', encoding='utf-8') as handle:
    for row in selection_manifest:
        handle.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n')
temporary_manifest.replace(manifest_path)
print('Selected scenes:', len(selected_examples))
print('Problem types:', Counter(example.problem_type for example in selected_examples))
print('Gold outcomes:', Counter(str(example.gold_semantic) for example in selected_examples))
print('Variants:', Counter(example.variant for example in selected_examples))
print('Orthogonal/layer control subset:', len(control_examples))


Select frozen causal scenes and one option permutation:  31%|###1      | 5/16 [00:00<?, ?cell/s]

Selected scenes: 304
Problem types: Counter({'multi_stack': 70, 'distance': 54, 'conflict_distance': 42, 'weight': 40, 'compatible_weight_distance': 40, 'conflict_weight': 40, 'conflict_balance': 12, 'balance': 6})
Gold outcomes: Counter({'left': 132, 'right': 131, 'balanced': 41})
Variants: Counter({'permutation_5': 61, 'permutation_1': 56, 'permutation_0': 53, 'permutation_3': 49, 'permutation_4': 44, 'permutation_2': 41})
Orthogonal/layer control subset: 72


In [8]:
%%cell_progress 7/16 Refit the frozen probe and construct intervention directions
cell_stage('Loading discovery scene means and frozen selection')
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

def torch_load_trusted(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')

selection = json.loads(
    discovery_selection_path.read_text(encoding='utf-8')
)
choice = selection['candidate_net_torque_layer']
SELECTED_LAYER_INDEX = int(choice['layer_index'])
FROZEN_ALPHA = float(choice['selected_alpha'])
assert SELECTED_LAYER_INDEX == spec['expected_layer']
assert abs(FROZEN_ALPHA - spec['expected_alpha']) < 1e-12
discovery = torch_load_trusted(discovery_activation_path)
layer_indices = [int(value) for value in discovery['layer_indices']]
layer_position = layer_indices.index(SELECTED_LAYER_INDEX)
X_fit = discovery['activations'][:, layer_position, :].float().numpy()
fit_metadata = pd.DataFrame(discovery['metadata'])
y_fit = pd.to_numeric(
    fit_metadata['net_torque'], errors='raise'
).to_numpy(dtype=float)

probe_scaler = StandardScaler()
probe = Ridge(alpha=FROZEN_ALPHA, solver='lsqr', max_iter=5000)
probe.fit(probe_scaler.fit_transform(X_fit), y_fit)
PROBE_MEAN = probe_scaler.mean_.astype(np.float32)
PROBE_SCALE = probe_scaler.scale_.astype(np.float32)
PROBE_COEF = np.asarray(probe.coef_, dtype=np.float32)
PROBE_INTERCEPT = float(probe.intercept_)
coefficient_norm_squared = float(np.dot(PROBE_COEF, PROBE_COEF))
assert coefficient_norm_squared > 0

TORQUE_SHIFT_UNIT = float(0.5 * np.std(y_fit, ddof=0))
torque_delta_z_unit = (
    TORQUE_SHIFT_UNIT * PROBE_COEF / coefficient_norm_squared
).astype(np.float32)
torque_delta_h_unit = (PROBE_SCALE * torque_delta_z_unit).astype(np.float32)
endpoint_z_norm = float(2.0 * np.linalg.norm(torque_delta_z_unit))

direction_rng = np.random.default_rng(SEED + SELECTED_LAYER_INDEX)
orthogonal_delta_h = {}
orthogonality_errors = []
for direction_index in range(ORTHOGONAL_DIRECTIONS):
    vector = direction_rng.normal(size=len(PROBE_COEF)).astype(np.float32)
    vector -= (
        float(np.dot(vector, PROBE_COEF)) / coefficient_norm_squared
    ) * PROBE_COEF
    vector_norm = float(np.linalg.norm(vector))
    assert vector_norm > 0
    endpoint_delta_z = endpoint_z_norm * vector / vector_norm
    orthogonality_errors.append(abs(float(np.dot(PROBE_COEF, endpoint_delta_z))))
    orthogonal_delta_h[direction_index] = (
        PROBE_SCALE * endpoint_delta_z
    ).astype(np.float32)
assert max(orthogonality_errors) < 1e-3

probe_direction_summary = {
    'selected_layer_index': SELECTED_LAYER_INDEX,
    'frozen_alpha': FROZEN_ALPHA,
    'hidden_size': X_fit.shape[1],
    'fit_scenes': len(y_fit),
    'training_torque_std': float(np.std(y_fit, ddof=0)),
    'torque_shift_unit': TORQUE_SHIFT_UNIT,
    'endpoint_decoded_torque_shift': 2.0 * TORQUE_SHIFT_UNIT,
    'endpoint_standardized_delta_norm': endpoint_z_norm,
    'probe_coefficient_norm': float(np.linalg.norm(PROBE_COEF)),
    'maximum_orthogonality_error': max(orthogonality_errors),
}
print(json.dumps(probe_direction_summary, indent=2))


Refit the frozen probe and construct intervention directions:  38%|###7      | 6/16 [00:00<?, ?cell/s]

{
  "selected_layer_index": 13,
  "frozen_alpha": 0.01,
  "hidden_size": 4096,
  "fit_scenes": 254,
  "training_torque_std": 8.808303535438725,
  "torque_shift_unit": 4.4041517677193625,
  "endpoint_decoded_torque_shift": 8.808303535438725,
  "endpoint_standardized_delta_norm": 4.479440212249756,
  "probe_coefficient_norm": 1.966385006904602,
  "maximum_orthogonality_error": 5.960464477539063e-08
}


In [9]:
%%cell_progress 8/16 Load the pinned model and decoder layers
cell_stage('Loading processor and pinned model weights')
adapter = HuggingFaceMultimodalAdapter(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=MODEL_DTYPE,
    device_map=DEVICE_MAP,
)
display(adapter.metadata)

cell_stage('Resolving decoder layers')
def resolve_module_path(root, path):
    value = root
    for component in path.split('.'):
        value = getattr(value, component)
    return value

def discover_decoder_layers(model):
    candidates = [
        discovery_identity.get('layer_path'),
        'model.language_model.layers',
        'model.text_model.layers',
        'language_model.model.layers',
        'language_model.layers',
        'model.model.layers',
        'model.layers',
    ]
    failures = []
    for path in candidates:
        if not path:
            continue
        try:
            layers = resolve_module_path(model, path)
        except (AttributeError, KeyError) as exc:
            failures.append(f'{path}: {exc}')
            continue
        if isinstance(layers, torch.nn.ModuleList) and len(layers) >= 8:
            return path, layers
        failures.append(f'{path}: {type(layers).__name__}')
    raise RuntimeError('Could not locate decoder layers:\n' + '\n'.join(failures))

LAYER_PATH, DECODER_LAYERS = discover_decoder_layers(adapter.model)
assert len(DECODER_LAYERS) == int(discovery_identity['decoder_layer_count'])
assert 1 <= SELECTED_LAYER_INDEX < len(DECODER_LAYERS) - 1
resolved_now = adapter.metadata.get('resolved_revision')
if resolved_now is not None:
    assert str(resolved_now) == MODEL_REVISION
print('Layer path:', LAYER_PATH)
print('Decoder layers:', len(DECODER_LAYERS))
print('Intervention layer:', SELECTED_LAYER_INDEX)
print('Neighbor controls:', SELECTED_LAYER_INDEX - 1, SELECTED_LAYER_INDEX + 1)


Load the pinned model and decoder layers:  44%|####3     | 7/16 [00:00<?, ?cell/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

{'adapter': 'HuggingFaceMultimodalAdapter',
 'model_id': 'Qwen/Qwen3.5-9B',
 'requested_revision': 'c202236235762e1c871ad0ccb60c8ee5ba337b9a',
 'resolved_revision': 'c202236235762e1c871ad0ccb60c8ee5ba337b9a',
 'dtype': 'bfloat16',
 'device_map': 'auto',
 'parameter_count': 9409813744,
 'torch_version': '2.11.0+cu128',
 'transformers_version': '5.14.0'}

Layer path: model.language_model.layers
Decoder layers: 32
Intervention layer: 13
Neighbor controls: 12 14


## 2. Hook intervention and resumable scoring

The hook clones the selected block output and changes only `hidden[:, -1, :]`.


In [10]:
%%cell_progress 9/16 Define safe hidden-state intervention scoring
cell_stage('Defining output replacement and semantic scoring')
import math
import os

def output_hidden_tensor(output):
    if torch.is_tensor(output):
        hidden = output
    elif isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        hidden = output[0]
    elif hasattr(output, 'last_hidden_state') and torch.is_tensor(
        output.last_hidden_state
    ):
        hidden = output.last_hidden_state
    else:
        raise TypeError(f'Unsupported decoder output: {type(output).__name__}')
    if hidden.ndim != 3:
        raise ValueError(f'Expected [batch, sequence, hidden], got {hidden.shape}')
    return hidden

def replace_output_hidden(output, modified):
    if torch.is_tensor(output):
        return modified
    if isinstance(output, tuple):
        return (modified,) + output[1:]
    if isinstance(output, list):
        return [modified] + output[1:]
    raise TypeError(
        'Decoder output object is not safely replaceable: '
        + type(output).__name__
    )

def choice_probabilities(scores):
    maximum = max(scores.values())
    weights = {key: math.exp(value - maximum) for key, value in scores.items()}
    normalizer = sum(weights.values())
    return {key: value / normalizer for key, value in weights.items()}

def decoded_torque_tensor(hidden):
    mean = torch.as_tensor(PROBE_MEAN, device=hidden.device, dtype=torch.float32)
    scale = torch.as_tensor(PROBE_SCALE, device=hidden.device, dtype=torch.float32)
    coefficient = torch.as_tensor(
        PROBE_COEF, device=hidden.device, dtype=torch.float32
    )
    standardized = (hidden.float() - mean) / scale
    return standardized @ coefficient + PROBE_INTERCEPT

def score_intervened_batch(examples, condition, delta_h):
    if not examples:
        return []
    if any(example.labels != examples[0].labels for example in examples):
        raise ValueError('All examples in a batch must share labels.')
    plan = adapter._candidate_plan(examples[0])
    if plan is None:
        raise RuntimeError('Causal scoring requires single-token A/B/C labels.')
    fixed_prefix, token_by_label = plan
    labels = list(examples[0].labels)
    conversations = [example.messages() for example in examples]
    inputs = adapter._append_text_prefix(
        adapter._to_device(
            adapter._template(conversations, add_generation_prompt=True)
        ),
        fixed_prefix,
    )
    target_layer_index = int(condition['target_layer_index'])
    target_layer = DECODER_LAYERS[target_layer_index]
    audit = {}

    def intervention_hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        modified = hidden.clone()
        delta = torch.as_tensor(
            delta_h, device=hidden.device, dtype=hidden.dtype
        )
        pre = hidden[:, -1, :]
        modified[:, -1, :] = pre + delta
        if target_layer_index == SELECTED_LAYER_INDEX:
            audit['decoded_pre'] = decoded_torque_tensor(pre).detach().cpu()
            audit['decoded_post'] = decoded_torque_tensor(
                modified[:, -1, :]
            ).detach().cpu()
            audit['hidden_norm'] = pre.float().norm(dim=1).detach().cpu()
            audit['delta_norm'] = (
                modified[:, -1, :].float() - pre.float()
            ).norm(dim=1).detach().cpu()
            audit['standardized_hidden_norm'] = (
                (pre.float() - torch.as_tensor(
                    PROBE_MEAN, device=pre.device
                )) / torch.as_tensor(PROBE_SCALE, device=pre.device)
            ).norm(dim=1).detach().cpu()
        return replace_output_hidden(output, modified)

    handle = target_layer.register_forward_hook(intervention_hook)
    started = time.perf_counter()
    try:
        with torch.inference_mode():
            output = adapter.model(**inputs, use_cache=False)
            next_logprobs = torch.log_softmax(
                output.logits[:, -1, :].float(), dim=-1
            )
    finally:
        handle.remove()
    latency = (time.perf_counter() - started) / len(examples)
    label_scores = torch.stack([
        next_logprobs[:, token_by_label[label]].detach().cpu()
        for label in labels
    ], dim=1)
    records = []
    for row_index, example in enumerate(examples):
        scores = {
            label: float(label_scores[row_index, column])
            for column, label in enumerate(labels)
        }
        probabilities = choice_probabilities(scores)
        semantic_scores = {
            str(example.label_to_semantic[label]): scores[label]
            for label in labels
        }
        semantic_probabilities = {
            str(example.label_to_semantic[label]): probabilities[label]
            for label in labels
        }
        predicted_label = max(sorted(labels), key=lambda key: scores[key])
        predicted_semantic = str(example.label_to_semantic[predicted_label])
        record = {
            'schema_version': 'causal-torque-record-1.0',
            'job_id': f"{condition['condition_id']}::{example.example_id}",
            'condition_id': condition['condition_id'],
            'condition_family': condition['condition_family'],
            'direction_id': condition['direction_id'],
            'target_layer_index': target_layer_index,
            'dose_multiplier': float(condition['dose_multiplier']),
            'planned_probe_shift': float(condition['planned_probe_shift']),
            'standardized_delta_norm': float(
                condition['standardized_delta_norm']
            ),
            'example_id': example.example_id,
            'scene_id': str(example.scene_id),
            'variant': example.variant,
            'problem_type': example.problem_type,
            'gold_outcome': str(example.gold_semantic),
            'net_torque': int(example.net_torque),
            'predicted_semantic': predicted_semantic,
            'correct': predicted_semantic == str(example.gold_semantic),
            'logprob_left': semantic_scores['left'],
            'logprob_right': semantic_scores['right'],
            'logprob_balanced': semantic_scores['balanced'],
            'prob_left': semantic_probabilities['left'],
            'prob_right': semantic_probabilities['right'],
            'prob_balanced': semantic_probabilities['balanced'],
            'left_right_margin': (
                semantic_scores['left'] - semantic_scores['right']
            ),
            'balanced_vs_best_direction_margin': (
                semantic_scores['balanced']
                - max(semantic_scores['left'], semantic_scores['right'])
            ),
            'latency_seconds': latency,
        }
        if target_layer_index == SELECTED_LAYER_INDEX:
            decoded_pre = float(audit['decoded_pre'][row_index])
            decoded_post = float(audit['decoded_post'][row_index])
            record.update({
                'decoded_torque_pre': decoded_pre,
                'decoded_torque_post': decoded_post,
                'realized_probe_shift': decoded_post - decoded_pre,
                'hidden_norm': float(audit['hidden_norm'][row_index]),
                'realized_delta_norm': float(
                    audit['delta_norm'][row_index]
                ),
                'standardized_hidden_norm': float(
                    audit['standardized_hidden_norm'][row_index]
                ),
            })
        else:
            record.update({
                'decoded_torque_pre': None,
                'decoded_torque_post': None,
                'realized_probe_shift': None,
                'hidden_norm': None,
                'realized_delta_norm': None,
                'standardized_hidden_norm': None,
            })
        records.append(record)
    return records

def chunked(values, size):
    for start in range(0, len(values), size):
        yield values[start:start + size]

print('Intervention scorer ready.')


Define safe hidden-state intervention scoring:  50%|#####     | 8/16 [00:00<?, ?cell/s]

Intervention scorer ready.


In [11]:
%%cell_progress 10/16 Run mandatory parity and perturbation smoke tests
cell_stage('Checking zero-dose hook parity')
zero_condition = {
    'condition_id': 'torque_dose_p0',
    'condition_family': 'torque_direction',
    'direction_id': 'torque',
    'target_layer_index': SELECTED_LAYER_INDEX,
    'dose_multiplier': 0.0,
    'planned_probe_shift': 0.0,
    'standardized_delta_norm': 0.0,
}
smoke_examples = selected_examples[:2]
zero_records = score_intervened_batch(
    smoke_examples, zero_condition, np.zeros_like(torque_delta_h_unit)
)
reference_scores = adapter.score_batch(smoke_examples)
for example, record, reference in zip(
    smoke_examples, zero_records, reference_scores
):
    semantic_to_label = {
        str(semantic): label
        for label, semantic in example.label_to_semantic.items()
    }
    for semantic in ('left', 'right', 'balanced'):
        label = semantic_to_label[semantic]
        difference = abs(
            record[f'logprob_{semantic}']
            - reference.logprob_by_label[label]
        )
        assert difference < 1e-3, (semantic, difference)

cell_stage('Checking realized positive and negative torque shifts')
smoke_shift_rows = []
for multiplier in (-2.0, 2.0):
    condition = {
        'condition_id': f'smoke_torque_{multiplier:+.0f}',
        'condition_family': 'torque_direction',
        'direction_id': 'torque',
        'target_layer_index': SELECTED_LAYER_INDEX,
        'dose_multiplier': multiplier,
        'planned_probe_shift': multiplier * TORQUE_SHIFT_UNIT,
        'standardized_delta_norm': abs(multiplier) * float(
            np.linalg.norm(torque_delta_z_unit)
        ),
    }
    records = score_intervened_batch(
        smoke_examples, condition, multiplier * torque_delta_h_unit
    )
    for record in records:
        tolerance = max(0.15, 0.03 * abs(condition['planned_probe_shift']))
        assert abs(
            record['realized_probe_shift']
            - condition['planned_probe_shift']
        ) < tolerance, record
        smoke_shift_rows.append(record)

smoke_frame = pd.DataFrame(smoke_shift_rows)
endpoint_ratio = (
    smoke_frame['realized_delta_norm'] / smoke_frame['hidden_norm']
)
standardized_ratio = (
    smoke_frame['standardized_delta_norm']
    / smoke_frame['standardized_hidden_norm']
)
assert float(standardized_ratio.max()) < 0.5, (
    'Endpoint perturbation exceeds 50% of standardized hidden norm; '
    'do not continue without a new versioned protocol.'
)
print('Zero-dose scoring parity: passed')
print('Endpoint decoded shift:', round(2 * TORQUE_SHIFT_UNIT, 4))
print('Raw delta/hidden norm range:', (
    round(float(endpoint_ratio.min()), 4),
    round(float(endpoint_ratio.max()), 4),
))
print('Standardized delta/hidden norm max:', round(
    float(standardized_ratio.max()), 4
))


Run mandatory parity and perturbation smoke tests:  56%|#####6    | 9/16 [00:00<?, ?cell/s]

AssertionError: {'schema_version': 'causal-torque-record-1.0', 'job_id': 'smoke_torque_-2::image_rule:q_multi_0a61a1f6f282dc5a_outcome_p3', 'condition_id': 'smoke_torque_-2', 'condition_family': 'torque_direction', 'direction_id': 'torque', 'target_layer_index': 13, 'dose_multiplier': -2.0, 'planned_probe_shift': -8.808303535438725, 'standardized_delta_norm': 4.479440212249756, 'example_id': 'image_rule:q_multi_0a61a1f6f282dc5a_outcome_p3', 'scene_id': 'multi_0a61a1f6f282dc5a', 'variant': 'permutation_3', 'problem_type': 'multi_stack', 'gold_outcome': 'left', 'net_torque': 11, 'predicted_semantic': 'left', 'correct': True, 'logprob_left': -0.41629624366760254, 'logprob_right': -1.9162962436676025, 'logprob_balanced': -1.6662962436676025, 'prob_left': 0.6624117938958737, 'prob_right': 0.14780404965619498, 'prob_balanced': 0.1897841564479314, 'left_right_margin': 1.5, 'balanced_vs_best_direction_margin': -1.25, 'latency_seconds': 0.21794486799998936, 'decoded_torque_pre': 16.995094299316

AssertionError: {'schema_version': 'causal-torque-record-1.0', 'job_id': 'smoke_torque_-2::image_rule:q_multi_0a61a1f6f282dc5a_outcome_p3', 'condition_id': 'smoke_torque_-2', 'condition_family': 'torque_direction', 'direction_id': 'torque', 'target_layer_index': 13, 'dose_multiplier': -2.0, 'planned_probe_shift': -8.808303535438725, 'standardized_delta_norm': 4.479440212249756, 'example_id': 'image_rule:q_multi_0a61a1f6f282dc5a_outcome_p3', 'scene_id': 'multi_0a61a1f6f282dc5a', 'variant': 'permutation_3', 'problem_type': 'multi_stack', 'gold_outcome': 'left', 'net_torque': 11, 'predicted_semantic': 'left', 'correct': True, 'logprob_left': -0.41629624366760254, 'logprob_right': -1.9162962436676025, 'logprob_balanced': -1.6662962436676025, 'prob_left': 0.6624117938958737, 'prob_right': 0.14780404965619498, 'prob_balanced': 0.1897841564479314, 'left_right_margin': 1.5, 'balanced_vs_best_direction_margin': -1.25, 'latency_seconds': 0.21794486799998936, 'decoded_torque_pre': 16.995094299316

In [ ]:
%%cell_progress 11/16 Build the complete frozen intervention job manifest
cell_stage('Constructing primary, orthogonal, and layer-control conditions')
condition_specs = []
condition_delta_h = {}
endpoint_unit_z_norm = float(np.linalg.norm(torque_delta_z_unit))
for multiplier in DOSE_MULTIPLIERS:
    label = str(multiplier).replace('-', 'm').replace('.', 'p')
    condition_id = f'torque_dose_{label}'
    condition_specs.append({
        'condition_id': condition_id,
        'condition_family': 'torque_direction',
        'direction_id': 'torque',
        'target_layer_index': SELECTED_LAYER_INDEX,
        'dose_multiplier': multiplier,
        'planned_probe_shift': multiplier * TORQUE_SHIFT_UNIT,
        'standardized_delta_norm': abs(multiplier) * endpoint_unit_z_norm,
        'sample_scope': 'all_selected_scenes',
    })
    condition_delta_h[condition_id] = multiplier * torque_delta_h_unit

for direction_index in range(ORTHOGONAL_DIRECTIONS):
    for sign in (-1.0, 1.0):
        sign_label = 'neg' if sign < 0 else 'pos'
        condition_id = f'orth_{direction_index:02d}_{sign_label}'
        condition_specs.append({
            'condition_id': condition_id,
            'condition_family': 'orthogonal_direction',
            'direction_id': f'orthogonal_{direction_index:02d}',
            'target_layer_index': SELECTED_LAYER_INDEX,
            'dose_multiplier': 2.0 * sign,
            'planned_probe_shift': 0.0,
            'standardized_delta_norm': endpoint_z_norm,
            'sample_scope': 'control_subset',
        })
        condition_delta_h[condition_id] = (
            sign * orthogonal_delta_h[direction_index]
        )

for relative_layer in (-1, 1):
    target_layer = SELECTED_LAYER_INDEX + relative_layer
    for sign in (-1.0, 1.0):
        sign_label = 'neg' if sign < 0 else 'pos'
        condition_id = f'neighbor_{target_layer}_{sign_label}'
        condition_specs.append({
            'condition_id': condition_id,
            'condition_family': 'neighbor_layer',
            'direction_id': f'selected_direction_at_layer_{target_layer}',
            'target_layer_index': target_layer,
            'dose_multiplier': 2.0 * sign,
            'planned_probe_shift': 2.0 * sign * TORQUE_SHIFT_UNIT,
            'standardized_delta_norm': endpoint_z_norm,
            'sample_scope': 'control_subset',
        })
        condition_delta_h[condition_id] = (
            2.0 * sign * torque_delta_h_unit
        )

condition_by_id = {
    condition['condition_id']: condition for condition in condition_specs
}
jobs = []
for condition in condition_specs:
    examples = (
        selected_examples
        if condition['sample_scope'] == 'all_selected_scenes'
        else control_examples
    )
    for example in examples:
        jobs.append({
            'job_id': f"{condition['condition_id']}::{example.example_id}",
            'condition_id': condition['condition_id'],
            'example_id': example.example_id,
        })
assert len({job['job_id'] for job in jobs}) == len(jobs)

job_manifest_path = OUTPUT_ROOT / 'job_manifest.jsonl'
temporary_jobs = job_manifest_path.with_suffix('.jsonl.tmp')
with temporary_jobs.open('w', encoding='utf-8') as handle:
    for job in jobs:
        handle.write(json.dumps(job, sort_keys=True) + '\n')
temporary_jobs.replace(job_manifest_path)
condition_path = OUTPUT_ROOT / 'conditions.json'
temporary_conditions = condition_path.with_suffix('.json.tmp')
temporary_conditions.write_text(
    json.dumps(condition_specs, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)
temporary_conditions.replace(condition_path)
print('Conditions:', len(condition_specs))
print('Expected intervention jobs:', len(jobs))
print('Primary jobs:', len(selected_examples) * len(DOSE_MULTIPLIERS))
print('Orthogonal jobs:', (
    ORTHOGONAL_DIRECTIONS * 2 * len(control_examples)
))
print('Neighbor-layer jobs:', 4 * len(control_examples))


In [ ]:
%%cell_progress 12/16 Capture or resume all causal intervention scores
cell_stage('Verifying append-only experiment identity')
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

def write_json_atomic(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False) + '\n',
        encoding='utf-8',
    )
    temporary.replace(path)

capture_identity = {
    'schema_version': 'causal-torque-capture-1.0',
    'model_id': MODEL_ID,
    'resolved_revision': MODEL_REVISION,
    'dtype': MODEL_DTYPE,
    'batch_size': BATCH_SIZE,
    'dataset_fingerprint': example_config.dataset_fingerprint(),
    'protocol_lock_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'discovery_selection_sha256': sha256_file(discovery_selection_path),
    'discovery_activations_sha256': sha256_file(discovery_activation_path),
    'selection_manifest_sha256': sha256_file(manifest_path),
    'job_manifest_sha256': sha256_file(job_manifest_path),
    'conditions_sha256': sha256_file(condition_path),
    'layer_path': LAYER_PATH,
    'decoder_layer_count': len(DECODER_LAYERS),
    'selected_layer_index': SELECTED_LAYER_INDEX,
    'frozen_alpha': FROZEN_ALPHA,
    'torque_shift_unit': TORQUE_SHIFT_UNIT,
    'expected_jobs': len(jobs),
    'seed': SEED,
}
capture_config_path = OUTPUT_ROOT / 'capture_config.json'
if capture_config_path.is_file():
    previous = json.loads(capture_config_path.read_text(encoding='utf-8'))
    assert previous.get('identity') == capture_identity, (
        'Existing causal capture belongs to a different experiment. '
        'Use a new versioned output directory.'
    )
else:
    write_json_atomic(capture_config_path, {
        'identity': capture_identity,
        'adapter_metadata': adapter.metadata,
        'probe_direction_summary': probe_direction_summary,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
    })

example_by_id = {example.example_id: example for example in selected_examples}
expected_job_ids = {job['job_id'] for job in jobs}
completed_job_ids = set()
shard_paths = sorted(SHARD_DIR.glob('batch_*.json'))
for path in tqdm(shard_paths, desc='Reading completed causal shards', unit='shard'):
    payload = json.loads(path.read_text(encoding='utf-8'))
    assert payload['capture_identity_sha256'] == sha256_file(capture_config_path)
    for record in payload['records']:
        job_id = str(record['job_id'])
        if job_id in completed_job_ids:
            raise RuntimeError(f'Duplicate completed job: {job_id}')
        completed_job_ids.add(job_id)
assert completed_job_ids <= expected_job_ids
print('Completed:', len(completed_job_ids), '| remaining:', (
    len(expected_job_ids - completed_job_ids)
))

next_shard = 0
if shard_paths:
    next_shard = max(
        int(path.stem.split('_')[-1]) for path in shard_paths
    ) + 1
completed_now = len(completed_job_ids)
progress = tqdm(
    total=len(expected_job_ids), initial=completed_now,
    desc='Causal intervention jobs', unit='job', dynamic_ncols=True,
)
for condition in condition_specs:
    condition_id = condition['condition_id']
    examples = (
        selected_examples
        if condition['sample_scope'] == 'all_selected_scenes'
        else control_examples
    )
    remaining_examples = [
        example for example in examples
        if f'{condition_id}::{example.example_id}' not in completed_job_ids
    ]
    if not remaining_examples:
        continue
    progress.set_postfix_str(condition_id, refresh=True)
    for batch in chunked(remaining_examples, BATCH_SIZE):
        records = score_intervened_batch(
            batch, condition, condition_delta_h[condition_id]
        )
        payload = {
            'schema_version': 'causal-torque-shard-1.0',
            'capture_identity_sha256': sha256_file(capture_config_path),
            'condition_id': condition_id,
            'records': records,
        }
        destination = SHARD_DIR / f'batch_{next_shard:06d}.json'
        temporary = SHARD_DIR / f'batch_{next_shard:06d}.json.tmp'
        temporary.write_text(
            json.dumps(payload, ensure_ascii=False) + '\n',
            encoding='utf-8',
        )
        temporary.replace(destination)
        next_shard += 1
        for record in records:
            completed_job_ids.add(record['job_id'])
        progress.update(len(records))
progress.close()
assert completed_job_ids == expected_job_ids
capture_summary = {
    'expected_jobs': len(expected_job_ids),
    'completed_jobs': len(completed_job_ids),
    'shard_count': len(list(SHARD_DIR.glob('batch_*.json'))),
    'selected_scenes': len(selected_examples),
    'control_scenes': len(control_examples),
}
write_json_atomic(OUTPUT_ROOT / 'capture_summary.json', capture_summary)
display(capture_summary)


## 3. Causal analysis

The model is released before analysis.


In [ ]:
%%cell_progress 13/16 Release GPU and assemble the intervention table
cell_stage('Releasing model weights')
import gc
del zero_records, reference_scores, smoke_shift_rows, smoke_frame
del adapter, DECODER_LAYERS
gc.collect()
torch.cuda.empty_cache()
print('GPU cache released.')

cell_stage('Reading and validating all atomic shards')
record_rows = []
seen_jobs = set()
for path in tqdm(
    sorted(SHARD_DIR.glob('batch_*.json')),
    desc='Loading causal shards', unit='shard',
):
    payload = json.loads(path.read_text(encoding='utf-8'))
    for record in payload['records']:
        if record['job_id'] in seen_jobs:
            raise RuntimeError(f"Duplicate job: {record['job_id']}")
        seen_jobs.add(record['job_id'])
        record_rows.append(record)
assert seen_jobs == expected_job_ids
intervention_records = pd.DataFrame(record_rows)
assert len(intervention_records) == len(jobs)
assert intervention_records['job_id'].is_unique

primary_records = intervention_records[
    intervention_records['condition_family'].eq('torque_direction')
].copy()
realized = primary_records[
    primary_records['dose_multiplier'].ne(0)
]
realized_error = (
    realized['realized_probe_shift'] - realized['planned_probe_shift']
).abs()
assert float(realized_error.quantile(0.99)) < max(
    0.2, 0.04 * (2 * TORQUE_SHIFT_UNIT)
)
orthogonal_audit = intervention_records[
    intervention_records['condition_family'].eq('orthogonal_direction')
]['realized_probe_shift'].abs()
assert float(orthogonal_audit.quantile(0.99)) < max(
    0.2, 0.04 * (2 * TORQUE_SHIFT_UNIT)
), 'Orthogonal controls changed decoded torque more than allowed.'
intervention_records.to_csv(
    ANALYSIS_DIR / 'intervention_records.csv', index=False
)
print('Records:', len(intervention_records))
print('99th percentile probe-shift audit error:', round(
    float(realized_error.quantile(0.99)), 5
))
print('99th percentile orthogonal decoded shift:', round(
    float(orthogonal_audit.quantile(0.99)), 5
))


In [ ]:
%%cell_progress 14/16 Analyze torque dose response and toward-gold correction
cell_stage('Computing per-scene dose-response effects')
from sklearn.metrics import accuracy_score

dose_rows = []
scene_effect_rows = []
for scene_id, group in primary_records.groupby('scene_id'):
    group = group.sort_values('dose_multiplier')
    assert group['dose_multiplier'].tolist() == list(DOSE_MULTIPLIERS)
    shifts = group['planned_probe_shift'].to_numpy(dtype=float)
    margins = group['left_right_margin'].to_numpy(dtype=float)
    slope = float(np.polyfit(shifts, margins, 1)[0])
    negative = group.iloc[0]
    baseline = group[group['dose_multiplier'].eq(0)].iloc[0]
    positive = group.iloc[-1]
    adjacent_increase_rate = float(np.mean(np.diff(margins) > 0))
    gold = str(baseline['gold_outcome'])
    row = {
        'scene_id': str(scene_id),
        'problem_type': str(baseline['problem_type']),
        'gold_outcome': gold,
        'net_torque': int(baseline['net_torque']),
        'margin_slope_per_torque_unit': slope,
        'endpoint_margin_effect': float(
            positive['left_right_margin'] - negative['left_right_margin']
        ),
        'adjacent_monotonic_increase_rate': adjacent_increase_rate,
        'baseline_predicted_semantic': str(baseline['predicted_semantic']),
        'baseline_correct': bool(baseline['correct']),
    }
    if gold in {'left', 'right'}:
        toward = positive if gold == 'left' else negative
        away = negative if gold == 'left' else positive
        sign = 1.0 if gold == 'left' else -1.0
        row.update({
            'toward_gold_correct': bool(toward['correct']),
            'away_from_gold_correct': bool(away['correct']),
            'toward_vs_baseline_gold_margin_effect': float(
                sign * (
                    toward['left_right_margin']
                    - baseline['left_right_margin']
                )
            ),
            'toward_vs_away_gold_margin_effect': float(
                sign * (
                    toward['left_right_margin']
                    - away['left_right_margin']
                )
            ),
            'baseline_error_corrected': bool(
                (not baseline['correct']) and toward['correct']
            ),
            'baseline_correct_harmed': bool(
                baseline['correct'] and (not toward['correct'])
            ),
        })
    else:
        row.update({
            'toward_gold_correct': None,
            'away_from_gold_correct': None,
            'toward_vs_baseline_gold_margin_effect': None,
            'toward_vs_away_gold_margin_effect': None,
            'baseline_error_corrected': None,
            'baseline_correct_harmed': None,
        })
        row['balance_peak_effect'] = float(
            baseline['prob_balanced']
            - 0.5 * (negative['prob_balanced'] + positive['prob_balanced'])
        )
    scene_effect_rows.append(row)

scene_effects = pd.DataFrame(scene_effect_rows)
for multiplier, frame in primary_records.groupby('dose_multiplier'):
    dose_rows.append({
        'dose_multiplier': float(multiplier),
        'planned_probe_shift': float(frame['planned_probe_shift'].iloc[0]),
        'scene_count': len(frame),
        'mean_left_right_margin': float(frame['left_right_margin'].mean()),
        'mean_prob_left': float(frame['prob_left'].mean()),
        'mean_prob_right': float(frame['prob_right'].mean()),
        'mean_prob_balanced': float(frame['prob_balanced'].mean()),
        'accuracy': float(frame['correct'].mean()),
    })
dose_response = pd.DataFrame(dose_rows).sort_values('dose_multiplier')

def stratified_indices(frame, rng):
    labels = (
        frame['problem_type'].astype(str)
        + '|' + frame['gold_outcome'].astype(str)
    ).to_numpy()
    parts = []
    for label in sorted(set(labels.tolist())):
        members = np.flatnonzero(labels == label)
        parts.append(rng.choice(members, size=len(members), replace=True))
    return np.concatenate(parts)

cell_stage('Running scene-level causal bootstrap')
nonbalanced = scene_effects[
    scene_effects['gold_outcome'].isin(['left', 'right'])
].reset_index(drop=True)
rng = np.random.default_rng(SEED + 5000)
bootstrap_rows = []
for replicate in tqdm(
    range(BOOTSTRAP_REPLICATES),
    desc='Primary causal scene bootstrap', unit='replicate',
):
    indices = stratified_indices(scene_effects, rng)
    nonbalanced_indices = stratified_indices(nonbalanced, rng)
    sampled = scene_effects.iloc[indices]
    sampled_nonbalanced = nonbalanced.iloc[nonbalanced_indices]
    bootstrap_rows.append({
        'replicate': replicate,
        'mean_endpoint_margin_effect': float(
            sampled['endpoint_margin_effect'].mean()
        ),
        'mean_slope': float(
            sampled['margin_slope_per_torque_unit'].mean()
        ),
        'positive_slope_rate': float(
            (sampled['margin_slope_per_torque_unit'] > 0).mean()
        ),
        'toward_vs_baseline_gold_margin_effect': float(
            sampled_nonbalanced[
                'toward_vs_baseline_gold_margin_effect'
            ].mean()
        ),
        'toward_vs_away_gold_margin_effect': float(
            sampled_nonbalanced[
                'toward_vs_away_gold_margin_effect'
            ].mean()
        ),
    })
primary_bootstrap = pd.DataFrame(bootstrap_rows)

def interval(series):
    low, high = np.quantile(np.asarray(series, dtype=float), [0.025, 0.975])
    return float(low), float(high)

primary_summary = {
    'selected_scene_count': len(scene_effects),
    'nonbalanced_scene_count': len(nonbalanced),
    'mean_endpoint_margin_effect': float(
        scene_effects['endpoint_margin_effect'].mean()
    ),
    'endpoint_effect_ci95': interval(
        primary_bootstrap['mean_endpoint_margin_effect']
    ),
    'mean_margin_slope_per_torque_unit': float(
        scene_effects['margin_slope_per_torque_unit'].mean()
    ),
    'slope_ci95': interval(primary_bootstrap['mean_slope']),
    'positive_slope_rate': float(
        (scene_effects['margin_slope_per_torque_unit'] > 0).mean()
    ),
    'positive_slope_rate_ci95': interval(
        primary_bootstrap['positive_slope_rate']
    ),
    'mean_toward_vs_baseline_gold_margin_effect': float(
        nonbalanced['toward_vs_baseline_gold_margin_effect'].mean()
    ),
    'toward_vs_baseline_ci95': interval(
        primary_bootstrap['toward_vs_baseline_gold_margin_effect']
    ),
    'mean_toward_vs_away_gold_margin_effect': float(
        nonbalanced['toward_vs_away_gold_margin_effect'].mean()
    ),
    'toward_vs_away_ci95': interval(
        primary_bootstrap['toward_vs_away_gold_margin_effect']
    ),
    'baseline_accuracy_nonbalanced': float(
        nonbalanced['baseline_correct'].mean()
    ),
    'toward_gold_accuracy_nonbalanced': float(
        nonbalanced['toward_gold_correct'].astype(bool).mean()
    ),
    'baseline_error_count_nonbalanced': int(
        (~nonbalanced['baseline_correct']).sum()
    ),
    'baseline_errors_corrected_count': int(
        nonbalanced['baseline_error_corrected'].astype(bool).sum()
    ),
    'baseline_error_correction_rate': float(
        nonbalanced.loc[
            ~nonbalanced['baseline_correct'], 'baseline_error_corrected'
        ].astype(bool).mean()
    ),
    'baseline_correct_harm_rate': float(
        nonbalanced.loc[
            nonbalanced['baseline_correct'], 'baseline_correct_harmed'
        ].astype(bool).mean()
    ),
}
balanced = scene_effects[scene_effects['gold_outcome'].eq('balanced')]
primary_summary['balanced_scene_count'] = len(balanced)
primary_summary['mean_balance_peak_effect'] = (
    float(balanced['balance_peak_effect'].mean()) if len(balanced) else None
)
scene_effects.to_csv(ANALYSIS_DIR / 'scene_effects.csv', index=False)
dose_response.to_csv(ANALYSIS_DIR / 'dose_response.csv', index=False)
primary_bootstrap.to_csv(
    ANALYSIS_DIR / 'primary_causal_bootstrap.csv', index=False
)
display(pd.Series(primary_summary))
display(dose_response.round(4))


In [ ]:
%%cell_progress 15/16 Analyze orthogonal and neighboring-layer controls
cell_stage('Comparing torque direction with 24 orthogonal directions')
control_ids = {str(example.scene_id) for example in control_examples}
torque_control = primary_records[
    primary_records['scene_id'].astype(str).isin(control_ids)
]
torque_control_effects = []
for scene_id, group in torque_control.groupby('scene_id'):
    negative = group[group['dose_multiplier'].eq(-2.0)].iloc[0]
    positive = group[group['dose_multiplier'].eq(2.0)].iloc[0]
    torque_control_effects.append(
        float(positive['left_right_margin'] - negative['left_right_margin'])
    )
torque_control_mean = float(np.mean(torque_control_effects))

orthogonal = intervention_records[
    intervention_records['condition_family'].eq('orthogonal_direction')
]
orthogonal_rows = []
for direction_id, direction_frame in orthogonal.groupby('direction_id'):
    scene_values = []
    for scene_id, group in direction_frame.groupby('scene_id'):
        negative = group[group['dose_multiplier'].lt(0)].iloc[0]
        positive = group[group['dose_multiplier'].gt(0)].iloc[0]
        scene_values.append(
            float(positive['left_right_margin'] - negative['left_right_margin'])
        )
    orthogonal_rows.append({
        'direction_id': direction_id,
        'scene_count': len(scene_values),
        'mean_endpoint_margin_effect': float(np.mean(scene_values)),
        'positive_scene_effect_rate': float(np.mean(np.asarray(scene_values) > 0)),
    })
orthogonal_summary = pd.DataFrame(orthogonal_rows).sort_values('direction_id')
orthogonal_p = float(
    (1 + (
        orthogonal_summary['mean_endpoint_margin_effect']
        >= torque_control_mean
    ).sum()) / (ORTHOGONAL_DIRECTIONS + 1)
)

cell_stage('Summarizing neighboring-layer controls')
neighbors = intervention_records[
    intervention_records['condition_family'].eq('neighbor_layer')
]
layer_rows = []
for target_layer, layer_frame in neighbors.groupby('target_layer_index'):
    effects = []
    for scene_id, group in layer_frame.groupby('scene_id'):
        negative = group[group['dose_multiplier'].lt(0)].iloc[0]
        positive = group[group['dose_multiplier'].gt(0)].iloc[0]
        effects.append(
            float(positive['left_right_margin'] - negative['left_right_margin'])
        )
    layer_rows.append({
        'target_layer_index': int(target_layer),
        'relative_to_selected': int(target_layer - SELECTED_LAYER_INDEX),
        'scene_count': len(effects),
        'mean_endpoint_margin_effect': float(np.mean(effects)),
        'positive_scene_effect_rate': float(np.mean(np.asarray(effects) > 0)),
        'effect_relative_to_selected_control_subset': float(
            np.mean(effects) / torque_control_mean
            if torque_control_mean != 0 else np.nan
        ),
    })
layer_control_summary = pd.DataFrame(layer_rows).sort_values(
    'target_layer_index'
)
control_summary = {
    'control_scene_count': len(control_ids),
    'selected_layer_torque_endpoint_effect': torque_control_mean,
    'orthogonal_direction_count': ORTHOGONAL_DIRECTIONS,
    'orthogonal_effect_mean': float(
        orthogonal_summary['mean_endpoint_margin_effect'].mean()
    ),
    'orthogonal_effect_max': float(
        orthogonal_summary['mean_endpoint_margin_effect'].max()
    ),
    'orthogonal_empirical_p_one_sided': orthogonal_p,
}
orthogonal_summary.to_csv(
    ANALYSIS_DIR / 'orthogonal_direction_summary.csv', index=False
)
layer_control_summary.to_csv(
    ANALYSIS_DIR / 'neighbor_layer_summary.csv', index=False
)
display(pd.Series(control_summary))
display(layer_control_summary.round(4))


In [ ]:
%%cell_progress 16/16 Save figures, causal decision, and cross-model comparison
cell_stage('Applying the frozen causal-support rule')
endpoint_ci_low = primary_summary['endpoint_effect_ci95'][0]
toward_away_ci_low = primary_summary['toward_vs_away_ci95'][0]
if (
    endpoint_ci_low > 0
    and toward_away_ci_low > 0
    and control_summary['orthogonal_empirical_p_one_sided'] <= 0.05
):
    causal_status = 'causal_directional_support'
elif endpoint_ci_low > 0 and toward_away_ci_low > 0:
    causal_status = 'directional_effect_without_orthogonal_specificity'
else:
    causal_status = 'causal_directional_support_not_established'

cell_stage('Rendering causal diagnostic figures')
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')

baseline_margin = float(
    dose_response.loc[
        dose_response['dose_multiplier'].eq(0), 'mean_left_right_margin'
    ].iloc[0]
)
figure, axis = plt.subplots(figsize=(8, 5.5))
axis.plot(
    dose_response['planned_probe_shift'],
    dose_response['mean_left_right_margin'] - baseline_margin,
    marker='o', linewidth=2,
)
axis.axhline(0, color='black', linestyle='--', linewidth=1)
axis.axvline(0, color='black', linestyle=':', linewidth=1)
axis.set_xlabel('Planned decoded-torque shift')
axis.set_ylabel('Mean change in log p(left) − log p(right)')
axis.set_title(f'{MODEL_ID}: torque-direction causal dose response')
plt.tight_layout()
plt.savefig(
    ANALYSIS_DIR / '01_causal_dose_response.png', dpi=180,
    bbox_inches='tight',
)
plt.show()

figure, axis = plt.subplots(figsize=(9, 5.5))
axis.hist(
    orthogonal_summary['mean_endpoint_margin_effect'], bins=12,
    color='0.7', edgecolor='white', label='24 orthogonal directions',
)
axis.axvline(
    torque_control_mean, color='tab:red', linewidth=2,
    label='torque direction',
)
axis.set_xlabel('Mean endpoint margin effect on control subset')
axis.set_ylabel('Direction count')
axis.set_title(f'{MODEL_ID}: torque direction versus norm-matched controls')
axis.legend()
plt.tight_layout()
plt.savefig(
    ANALYSIS_DIR / '02_orthogonal_controls.png', dpi=180,
    bbox_inches='tight',
)
plt.show()

problem_summary = scene_effects.groupby('problem_type').agg(
    scene_count=('scene_id', 'size'),
    mean_endpoint_effect=('endpoint_margin_effect', 'mean'),
    positive_slope_rate=(
        'margin_slope_per_torque_unit', lambda values: float((values > 0).mean())
    ),
    baseline_accuracy=('baseline_correct', 'mean'),
).reset_index().sort_values('problem_type')
problem_summary.to_csv(
    ANALYSIS_DIR / 'causal_effect_by_problem_type.csv', index=False
)
figure, axis = plt.subplots(figsize=(11, 5.5))
axis.bar(
    problem_summary['problem_type'], problem_summary['mean_endpoint_effect']
)
axis.axhline(0, color='black', linestyle='--', linewidth=1)
axis.set_ylabel('Mean endpoint left-right margin effect')
axis.set_title(f'{MODEL_ID}: causal effect by problem type')
axis.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.savefig(
    ANALYSIS_DIR / '03_effect_by_problem_type.png', dpi=180,
    bbox_inches='tight',
)
plt.show()

causal_summary = {
    'schema_version': 'causal-torque-analysis-1.0',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'analysis_status': protocol_identity['analysis_status'],
    'model_id': MODEL_ID,
    'resolved_revision': MODEL_REVISION,
    'selected_layer_index': SELECTED_LAYER_INDEX,
    'frozen_alpha': FROZEN_ALPHA,
    'probe_direction': probe_direction_summary,
    'primary': primary_summary,
    'controls': control_summary,
    'neighbor_layers': layer_control_summary.to_dict(orient='records'),
    'causal_status': causal_status,
    'causal_support_rule': protocol_identity['causal_support_rule'],
    'interpretation_limit': protocol_identity['interpretation_limit'],
}
write_json_atomic(ANALYSIS_DIR / 'causal_summary.json', causal_summary)

required_outputs = [
    PROTOCOL_LOCK_PATH,
    manifest_path,
    job_manifest_path,
    condition_path,
    capture_config_path,
    OUTPUT_ROOT / 'capture_summary.json',
    ANALYSIS_DIR / 'intervention_records.csv',
    ANALYSIS_DIR / 'scene_effects.csv',
    ANALYSIS_DIR / 'dose_response.csv',
    ANALYSIS_DIR / 'primary_causal_bootstrap.csv',
    ANALYSIS_DIR / 'orthogonal_direction_summary.csv',
    ANALYSIS_DIR / 'neighbor_layer_summary.csv',
    ANALYSIS_DIR / 'causal_effect_by_problem_type.csv',
    ANALYSIS_DIR / 'causal_summary.json',
    ANALYSIS_DIR / '01_causal_dose_response.png',
    ANALYSIS_DIR / '02_orthogonal_controls.png',
    ANALYSIS_DIR / '03_effect_by_problem_type.png',
]
missing = [str(path) for path in required_outputs if not path.is_file()]
assert not missing, 'Missing outputs:\n' + '\n'.join(missing)
run_manifest = {
    'schema_version': 'causal-torque-run-1.0',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_id': MODEL_ID,
    'causal_status': causal_status,
    'protocol_lock_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'capture_config_sha256': sha256_file(capture_config_path),
    'required_outputs': [str(path) for path in required_outputs],
}
write_json_atomic(ANALYSIS_DIR / 'run_manifest.json', run_manifest)

cell_stage('Checking whether both causal model runs are complete')
comparison_paths = {
    'Qwen': (
        PROJECT_ROOT / 'results' / 'Qwen__Qwen3.5-9B'
        / 'causal_torque_intervention_v1' / 'analysis' / 'causal_summary.json'
    ),
    'Gemma': (
        PROJECT_ROOT / 'results' / 'google__gemma-4-12B-it'
        / 'causal_torque_intervention_v1' / 'analysis' / 'causal_summary.json'
    ),
}
if all(path.is_file() for path in comparison_paths.values()):
    rows = []
    for model_name, path in comparison_paths.items():
        summary = json.loads(path.read_text(encoding='utf-8'))
        rows.append({
            'model': model_name,
            'causal_status': summary['causal_status'],
            'mean_endpoint_margin_effect': (
                summary['primary']['mean_endpoint_margin_effect']
            ),
            'endpoint_ci95_low': (
                summary['primary']['endpoint_effect_ci95'][0]
            ),
            'mean_toward_vs_away_gold_margin_effect': (
                summary['primary'][
                    'mean_toward_vs_away_gold_margin_effect'
                ]
            ),
            'baseline_error_correction_rate': (
                summary['primary']['baseline_error_correction_rate']
            ),
            'orthogonal_empirical_p': (
                summary['controls']['orthogonal_empirical_p_one_sided']
            ),
        })
    comparison = pd.DataFrame(rows)
    comparison_root = (
        PROJECT_ROOT / 'results' / 'causal_torque_comparison_v1'
    )
    comparison_root.mkdir(parents=True, exist_ok=True)
    comparison.to_csv(
        comparison_root / 'causal_model_comparison.csv', index=False
    )
    display(comparison.round(4))
    print('Cross-model causal comparison:', comparison_root)
else:
    missing_models = [
        name for name, path in comparison_paths.items() if not path.is_file()
    ]
    print('Cross-model comparison deferred. Missing:', missing_models)
    print('Run the other model in a fresh A100 runtime.')

print('Causal intervention complete:', MODEL_ID)
print('Status:', causal_status)
print('Interpretation limit:', protocol_identity['interpretation_limit'])
